# Playbook · Backtest without look-ahead or survivorship bias

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/playbooks/backtest-without-bias.ipynb)

**Who it is for:** quants and systematic researchers building a factor backtest.

**What you get:** a point-in-time panel (members as of each rebalance date, ratios as known that day), a place to join your own prices on CIK, and a receipt for any number you use.

**Plan needed:** none for every cell (free sample, no key). One extra check reaches back to 2008 and needs a free Benchmark key: it prints a one-line skip without one.

Valuein serves what companies reported to the SEC, not market prices: bring your own prices and join them on the 10-digit CIK.

## Step 1 · Open the client, no key needed

Every cell runs on the free sample (S&P 500 companies, current and former, last five years) with no account and no card, through the same tables and code your plan uses.

In [1]:
%pip install -q "valuein-sdk>=1.1.0" matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
from datetime import datetime, timezone

import pandas as pd

from valuein_sdk import ValueinClient

warnings.filterwarnings("ignore", message="Defaulting to SAMPLE dataset")
client = ValueinClient()  # no key: the free sample. VALUEIN_API_KEY switches to your plan.
print(f"plan: {client.plan!r}")

plan: 'full'


**Check it worked:** it prints `plan: 'sample'` with no key (a free Benchmark key prints `'sp500'`, Institutional `'full'`).

## Step 2 · Build the universe on each rebalance date

Today's index list is the wrong universe for a past date: it leaves out every company that later failed, merged or was removed. `index_removal_date` is set for members that left afterwards; keep them.

In [3]:
REBALANCE_DATES = ["2024-10-31", "2025-10-31"]

members = client.universe("SP500", dates=REBALANCE_DATES)
print(members.groupby("rebalance_date")["cik"].nunique())
print(members.loc[members["index_removal_date"].notna(), "cik"].nunique(),
      "of them have left the index since")

rebalance_date
2024-10-31    500
2025-10-31    499
Name: cik, dtype: int64
44 of them have left the index since


In [4]:
# Membership reaches back to 1996 (reconstructed, incomplete before about 2007; each row has a confidence).
try:
    m2008 = client.pit_universe("2008-06-30")
    print(m2008["cik"].nunique(), "members on 2008-06-30")
    print(m2008.loc[m2008["cik"] == "0000806085", ["cik", "company_name", "ticker_at_date"]].to_string(index=False))
except Exception as exc:  # a plan or network error must not stop the notebook
    print(f"Skipped: {exc}")

500 members on 2008-06-30
       cik                             company_name ticker_at_date
0000806085 LEHMAN BROTHERS HOLDINGS INC. PLAN TRUST          LEHMQ


In [5]:
# Fundamentals for 2008 are outside the free sample: they need a free Benchmark key.
if client.plan == "sample":
    print("Skipped: set VALUEIN_API_KEY (free Benchmark key: https://valuein.biz/signup/free) to read 2008 filings.")
else:
    try:
        with ValueinClient(as_of=datetime(2008, 6, 30, tzinfo=timezone.utc)) as past:
            lehman = past.run_query("""
                SELECT numeric_value / 1e9 AS total_assets_bn, period_end, accession_id, accepted_at
                FROM fact
                WHERE entity_id = '0000806085' AND standard_concept = 'TotalAssets'
                QUALIFY ROW_NUMBER() OVER (ORDER BY period_end DESC, accepted_at DESC, priority DESC) = 1
            """)
        print(lehman.to_string(index=False))
    except Exception as exc:
        print(f"Skipped: {exc}")

 total_assets_bn period_end         accession_id               accepted_at
         786.035 2008-02-29 0001104659-08-023292 2008-04-08 23:38:10+00:00


**Check it worked:** one member count per date; the 2008 cell shows Lehman Brothers (CIK 0000806085), a member on 2008-06-30 that was delisted later; the last cell prints a skip without a key.

## Step 3 · Pull fundamentals as they were known

A fiscal year ends weeks before its 10-K is public, so filter on when the SEC accepted the filing (`accepted_at`), never on the period it describes (`period_end`). Setting `as_of` does this for the whole session.

In [6]:
QUESTION = """
    SELECT numeric_value / 1e9 AS revenue_bn, accession_id, accepted_at
    FROM fact
    WHERE entity_id = '{cik}'
      AND standard_concept = 'TotalRevenue'
      AND fiscal_period = 'FY'
    QUALIFY ROW_NUMBER() OVER (ORDER BY period_end DESC, accepted_at DESC, priority DESC) = 1
"""
cik = client.resolve("AAPL")["cik"].iloc[0]

answers = []
for day in [datetime(2025, 10, 30, tzinfo=timezone.utc), datetime(2025, 11, 1, tzinfo=timezone.utc)]:
    with ValueinClient(as_of=day) as past:  # as_of must be timezone-aware
        row = past.run_query(QUESTION.format(cik=cik)).iloc[0]
    answers.append({"as of": f"{day:%Y-%m-%d}", **row.to_dict()})
pd.DataFrame(answers)

,as of,revenue_bn,accession_id,accepted_at
0,2025-10-30,391.035,0000320193-24-000123,2024-11-01 10:01:36+00:00
1,2025-11-01,416.161,0000320193-25-000079,2025-10-31 10:01:26+00:00


**Check it worked:** Apple's latest annual revenue is 391.035 (USD billions) as of 2025-10-30 and 416.161 as of 2025-11-01, accession 0000320193-25-000079 on the second row: the 10-K was accepted on 2025-10-31.

## Step 4 · Attach the factors known on each date

A screen run on a date can only use ratios that were public by then. `signal_panel` attaches the latest ratio vintage accepted by the end of each rebalance day; unknown values stay NaN and no member is dropped.

In [7]:
RATIOS = ["return_on_equity", "net_margin", "debt_to_equity"]

panel = client.signal_panel(members, ratios=RATIOS)
print(len(panel), "member-dates;", panel["return_on_equity"].notna().sum(), "with a known ROE")
panel[["rebalance_date", "cik", "ticker", *RATIOS]].head(3)

999 member-dates; 982 with a known ROE


,rebalance_date,cik,ticker,return_on_equity,net_margin,debt_to_equity
0,2024-10-31,0001090872,A,0.222422,0.181472,0.467921
1,2024-10-31,0000320193,AAPL,1.719495,0.253062,1.787533
2,2024-10-31,0001551152,ABBV,0.349098,0.089528,5.711744


**Check it worked:** the panel has one row per member per date, with the three ratios and `NaN` where nothing was known yet.

## Step 5 · Join your own prices on CIK

Valuein does not serve market prices. Bring your closes from your own provider and key them on the 10-digit CIK, which stays with a company for life. `price_multiples` pairs them with filings accepted before 16:00 New York time on each price date.

In [8]:
# EXAMPLE VALUES ONLY so the cell runs: replace with your own closes,
# for example pd.read_parquet("my_closes.parquet"). They are not market data.
MY_CLOSES = pd.DataFrame({
    "cik": ["0000320193", "0000320193"],
    "price_date": ["2025-10-30", "2025-10-31"],
    "close": [200.00, 200.00],
    "currency": ["USD", "USD"],
    "source_label": ["example value", "example value"],
})

out = client.price_multiples(MY_CLOSES, metrics=["market_cap", "pe"])
print(out[["cik", "price_date", "close", "pe", "fundamentals_accepted_at",
           "fundamentals_accession_id"]].to_string(index=False))

       cik price_date  close         pe  fundamentals_accepted_at fundamentals_accession_id
0000320193 2025-10-30  200.0  32.894737 2024-11-01 10:01:36+00:00      0000320193-24-000123
0000320193 2025-10-31  200.0  26.809651 2025-10-31 10:01:26+00:00      0000320193-25-000079


In [9]:
panel["rebalance_date"] = pd.to_datetime(panel["rebalance_date"])
closes = MY_CLOSES.assign(rebalance_date=pd.to_datetime(MY_CLOSES["price_date"]))

backtest_input = panel.merge(closes[["cik", "rebalance_date", "close"]],
                             on=["cik", "rebalance_date"], how="left")
print(backtest_input["close"].notna().sum(), "of", len(backtest_input),
      "member-dates priced; unpriced rows are kept")

1 of 999 member-dates priced; unpriced rows are kept


**Check it worked:** the P/E on 2025-10-30 rests on the older 10-K and the one on 2025-10-31 on accession 0000320193-25-000079 (the filing is accepted at 10:01 UTC, before the 16:00 New York cut); unpriced member-dates stay in the panel.

## Step 6 · Trace a figure to its filing

Before you trust a backtest, pull one number and follow it to the SEC filing it came from. Every figure carries a `fact_id`.

In [10]:
print(out.iloc[1]["input_fact_ids"])  # the filed inputs behind the P/E on 2025-10-31

{'revenue': '97960097ffc0ea39388b21822abea04be91df63698bec0422f8e4327b0a5247b', 'net_income': '9b68163632e6f1ea651ff5091c478c6e694a764b51a5eef6d1f2eb053822782d', 'eps': 'cb7bee482a97d98fabb30ee3e7e30085d0be67b0914a9d6b7b7adb2e0569e3a6', 'operating_income': '6c17abcdc07f8c91ebfa38ac0bb15bedb3b38bf91f0815d6ba14a0d4f58aae33', 'da': 'ad6466cbf4f4b0691530367393af8b08f6847332b5cdcc3099c6b9bb6741f218', 'dps': '2dcb3bd410c6281fade886690bfaedcde532e4ba99e86f3d96b71ef47c555ec6', 'equity': 'c41eae6d24a64a3f4d23983ea78bdb36396faa6009b2006be8eba5d9b7bab23b', 'cash': 'f18a0a5f90b307e44f3d5cb44d825354c9188ec008a6a10101f63fa56783de00', 'sti': 'b9693e2aad8f680af7cfd260064396f82446c853126800c5308041c760e82228', 'shares': 'e05b6bdda0b7211bcf33e02608bdcad8e542caceb8b0f53a8ccc48683db2d587'}


In [11]:
trace = client.run_query(f"""
    SELECT fact_id, accession_id, period_end, numeric_value / 1e9 AS revenue_bn, accepted_at
    FROM fact
    WHERE entity_id = '{cik}'
      AND standard_concept = 'TotalRevenue'
      AND fiscal_period = 'FY'
      AND period_end = DATE '2025-09-27'
    ORDER BY accepted_at, priority DESC
    LIMIT 1
""")
print(trace.iloc[0].to_string())
client.close()

fact_id         97960097ffc0ea39388b21822abea04be91df63698bec0...
accession_id                                 0000320193-25-000079
period_end                                    2025-09-27 00:00:00
revenue_bn                                                416.161
accepted_at                             2025-10-31 10:01:26+00:00


**Check it worked:** the row for the period ended 2025-09-27 shows revenue 416.161 (USD billions), accession 0000320193-25-000079 and its `fact_id`.

**Next:** [Smart money, point in time](smart-money-point-in-time.ipynb) · [Verify a number](../notebooks/08_verify_a_number.ipynb)

_Index names are trademarks of their owners; Valuein is not affiliated with or endorsed by any index provider._